# AC Sparta Praha — Matchday 1 Analysis

Chance Liga 2026/27, matchday 1 (2026-07-25, Stadion Za Lužánkami, Brno).
Zbrojovka Brno won 3-1; Sparta's goal came from J. Mercado.

Combines `match_data.py` + `build_charts.py` + `build_pdf.py` (same folder) into one
runnable notebook. Rebuilds the page set of a season-long "SPL" template the user
supplied (yellow/green scheme, full CZ 2025/26 season, all-16-team percentiles) in
Marc Lamberts' Meridian house style (dark mode), but scoped to what this repo
actually has: matchday-1 event feeds for 14 of the CZ 2026-2027 season's 16 teams.
Every "league ranking" page is built from that 14-team, single-match sample and
says so in its dek — never presented as season-level data.

Run all cells top to bottom to regenerate `Visuals/*.png` and the compiled PDF.

## Setup

In [1]:
import json
import math
import os

# Notebook-safe path resolution (no __file__ inside a notebook cell) --
# assumes this notebook is opened/run from its own directory, same as
# REPO_ROOT in the original match_data.py.
NOTEBOOK_DIR = os.path.abspath("")
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(NOTEBOOK_DIR)))
EVENTS_DIR = os.path.join(REPO_ROOT, "CZ Events", "CZ 2026-2027")

SPARTA_ID = "5ocdn3a6s75u0d0dy0rbou0xc"
BRNO_ID = "6k350zwynsc23f0sxw9akgc6y"

COMPETITION = "Chance Liga 2026/27, Matchday 1"
VENUE = "Stadion Za Lužánkami, Brno"
MATCH_DATE = "2026-07-25"
SOURCE = "Opta event data (CZ 2026-27, matchday 1, 14 of 16 teams) + own xG model"

# All matchday-1 fixtures this repo has an event feed for -- (file, home_id,
# home_name, away_id, away_name). Doubles as the "league sample" for every
# ranking page: 14 teams, one match each, explicitly not a season.
MATCHES = [
    dict(path="2026-07-25_FC Viktoria Plzeň - FC Slovan Liberec.json",
         home_id="c6fx1460nlkawjgh67sp7a1hd", home_name="Viktoria Plzeň",
         away_id="2c4rs2vp0tyjiqa7y3gfttf24", away_name="Slovan Liberec"),
    dict(path="2026-07-25_FC Zbrojovka Brno - AC Sparta Praha.json",
         home_id=BRNO_ID, home_name="Zbrojovka Brno",
         away_id=SPARTA_ID, away_name="Sparta Praha"),
    dict(path="2026-07-25_FC Zlín - FC Baník Ostrava.json",
         home_id="aj1nbeiatqrs6e47mnhjidn15", home_name="Zlín",
         away_id="dfvvrv84skv23rsn1k6kt4slc", away_name="Baník Ostrava"),
    dict(path="2026-07-25_FK Teplice - Bohemians Praha 1905.json",
         home_id="41eivtin75c5fu33x3zfx956b", home_name="Teplice",
         away_id="bqcrqg0367eqzrt4vjb5apu6g", away_name="Bohemians 1905"),
    dict(path="2026-07-26_FC Hradec Králové - FK Pardubice.json",
         home_id="1v75g4bk8vzrvu0jmaro6lila", home_name="Hradec Králové",
         away_id="4xbgquadoen1b303u4hi9nhg9", away_name="Pardubice"),
    dict(path="2026-07-26_FK Jablonec - SK Sigma Olomouc.json",
         home_id="bdz8tx20ekj1ryi2e2u13jdl", home_name="Jablonec",
         away_id="dchxm00ei80l8ljbcfpdill8k", away_name="Sigma Olomouc"),
    dict(path="2026-07-26_SK Slavia Praha - 1. FC Slovácko.json",
         home_id="8kpapuorr6hf0vosnovbreqqd", home_name="Slavia Praha",
         away_id="bp5x8iw8pstucx6s4iqht6xqf", away_name="Slovácko"),
]

TEAM_NAMES = {}
TEAM_MATCH_FILE = {}
for m in MATCHES:
    TEAM_NAMES[m["home_id"]] = m["home_name"]
    TEAM_NAMES[m["away_id"]] = m["away_name"]
    TEAM_MATCH_FILE[m["home_id"]] = m
    TEAM_MATCH_FILE[m["away_id"]] = m

X_SCALE, Y_SCALE = 1.05, 0.68     # Opta 0-100 units -> metres (105 x 68 pitch)
GOAL_X = 105.0
GOAL_Y = 34.0
GOAL_WIDTH = 7.32
PITCH_X, PITCH_Y = 105.0, 68.0

T_PASS, T_TAKE_ON, T_FOUL, T_OUT = 1, 3, 4, 5
T_CORNER_AWARDED = 6
T_TACKLE, T_INTERCEPTION = 7, 8
T_CLEARANCE = 12
T_MISS, T_ATTEMPT_SAVED, T_GOAL, T_POST = 13, 15, 16, 14
T_CARD = 17
T_SUB_OFF, T_SUB_ON = 18, 19
T_CHALLENGE = 45
T_AERIAL = 44
T_BALL_RECOVERY, T_DISPOSSESSED = 49, 50
T_BLOCKED_PASS = 74

SHOT_TYPES = {T_MISS, T_POST, T_ATTEMPT_SAVED, T_GOAL}
DEFENSIVE_TYPES = {T_TACKLE: "Tackle", T_INTERCEPTION: "Interception", T_CLEARANCE: "Clearance"}
PRESSING_TYPES = {T_TACKLE: "Tackle", T_INTERCEPTION: "Interception", T_CHALLENGE: "Challenge"}

Q_LONG_BALL = 1
Q_CROSS, Q_THROUGH, Q_FREE_KICK, Q_CORNER, Q_THROW_IN = 2, 3, 5, 6, 107
Q_HEAD = 15
Q_RIGHT_FOOT, Q_LEFT_FOOT = 20, 72
Q_END_X, Q_END_Y = 140, 141
Q_ZONE = 56
Q_REGULAR_PLAY, Q_FAST_BREAK, Q_SET_PIECE, Q_FROM_CORNER = 22, 23, 24, 25
Q_BIG_CHANCE = 80
Q_YELLOW_CARD, Q_SECOND_YELLOW, Q_RED_CARD = 31, 32, 33
Q_GOAL_KICK = 124
Q_CUTBACK = 195

SET_PIECE_QIDS = {Q_FREE_KICK, Q_CORNER, Q_THROW_IN}
WIDE_THIRD = 100 / 3
SWITCH_MIN_DIST_M = 30

PPDA_ZONE_M = 63.0

ZONE14 = (70.0, 88.5, 27.2, 40.8)              # x0, x1, y0, y1
HALF_SPACES = [(52.5, 105.0, 13.6, 27.2), (52.5, 105.0, 40.8, 54.4)]
BOX_Y = (13.84, 54.16)

## Data loading + parsing (`match_data.py`)

Opta MA3 event feed, same typeId/qualifierId conventions as the rest of this repo.
New metric definitions (goal kick, cutback, switch of play) are sourced from
existing modules elsewhere in this repo rather than invented fresh — see the
module docstring.

In [2]:
def qmap(e):
    return {q["qualifierId"]: q.get("value") for q in e.get("qualifier", []) or []}


def has_q(e, qid):
    return any(q["qualifierId"] == qid for q in e.get("qualifier", []) or [])


def event_time(e):
    return e["timeMin"] * 60 + e["timeSec"]


def load_match(filename):
    with open(os.path.join(EVENTS_DIR, filename), encoding="utf-8") as f:
        data = json.load(f)
    events = data["event"]
    events.sort(key=lambda e: (e["periodId"], event_time(e), e["eventId"]))
    return data["matchDetails"], events


def team_name(cid):
    return TEAM_NAMES.get(cid, cid)


def to_m(x, y):
    return x * X_SCALE, y * Y_SCALE


def compute_attack_directions(events):
    sums = {}
    for e in events:
        if e["typeId"] != T_PASS or e.get("x") is None:
            continue
        if e["x"] == 0 and e["y"] == 0:
            continue
        key = (e["contestantId"], e["periodId"])
        s = sums.setdefault(key, [0.0, 0])
        s[0] += e["x"]
        s[1] += 1
    return {key: (1 if (total / n if n else 50) < 50 else -1) for key, (total, n) in sums.items()}


def norm_xy(e, directions):
    d = directions.get((e["contestantId"], e["periodId"]), 1)
    x, y = e["x"], e["y"]
    if d == 1:
        return x, y
    return 100.0 - x, 100.0 - y


def shot_angle_deg(x_m, y_m):
    dx = GOAL_X - x_m
    if dx <= 0:
        return 0.0
    y1 = y_m - (GOAL_Y - GOAL_WIDTH / 2)
    y2 = y_m - (GOAL_Y + GOAL_WIDTH / 2)
    denom = dx * dx + y1 * y2
    a = math.atan2(GOAL_WIDTH * dx, denom) if denom != 0 else math.pi / 2
    if a < 0:
        a += math.pi
    return math.degrees(a)


def shot_xg(x_m, y_m, is_header):
    dist = math.hypot(GOAL_X - x_m, GOAL_Y - y_m)
    angle = shot_angle_deg(x_m, y_m)
    z = -2.0 + 3.6 * math.radians(angle) - 0.085 * dist - (0.65 if is_header else 0.0)
    xg = 1.0 / (1.0 + math.exp(-z))
    return max(0.015, min(0.94, xg))


def xt_value(x100, y100):
    x_m, y_m = to_m(x100, y100)
    return shot_xg(x_m, y_m, is_header=False)


def build_shots(events, directions):
    rows = []
    for e in events:
        if e["typeId"] not in SHOT_TYPES or e.get("x") is None:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        is_header = has_q(e, Q_HEAD)
        xg = shot_xg(xm, ym, is_header)
        outcome = {T_GOAL: "Goal", T_ATTEMPT_SAVED: "Saved", T_MISS: "Miss", T_POST: "Post"}[e["typeId"]]
        if has_q(e, Q_FROM_CORNER):
            situation = "Corner"
        elif has_q(e, Q_SET_PIECE):
            situation = "Set piece"
        elif has_q(e, Q_FAST_BREAK):
            situation = "Fast break"
        else:
            situation = "Open play"
        rows.append({
            "contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"), "eventId": e["eventId"],
            "minute": e["timeMin"], "period": e["periodId"], "x": xm, "y": ym,
            "outcome": outcome, "on_target": e["typeId"] in (T_GOAL, T_ATTEMPT_SAVED),
            "is_goal": e["typeId"] == T_GOAL, "is_header": is_header,
            "big_chance": has_q(e, Q_BIG_CHANCE), "situation": situation, "xg": xg,
        })
    return rows


def build_passes(events, directions):
    rows = []
    for e in events:
        if e["typeId"] != T_PASS or e.get("x") is None:
            continue
        q = qmap(e)
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        completed = e["outcome"] == 1
        end_x = end_y = ex = ey = None
        if Q_END_X in q and Q_END_Y in q:
            d = directions.get((e["contestantId"], e["periodId"]), 1)
            ex, ey = float(q[Q_END_X]), float(q[Q_END_Y])
            if d == -1:
                ex, ey = 100.0 - ex, 100.0 - ey
            end_x, end_y = to_m(ex, ey)
        start_dist = math.hypot(GOAL_X - xm, GOAL_Y - ym)
        end_dist = math.hypot(GOAL_X - end_x, GOAL_Y - end_y) if end_x is not None else None
        progressive = (completed and end_dist is not None and
                       end_dist <= start_dist * 0.75 and end_x > xm)
        xt_start = xt_value(x, y)
        xt_end = xt_value(ex, ey) if ex is not None else None
        xt_added = (xt_end - xt_start) if (completed and xt_end is not None) else 0.0

        is_switch = False
        if not (SET_PIECE_QIDS & set(q.keys())) and ex is not None:
            raw_y0, raw_y1 = e["y"], float(q[Q_END_Y])
            left0, right0 = raw_y0 <= WIDE_THIRD, raw_y0 >= (100 - WIDE_THIRD)
            left1, right1 = raw_y1 <= WIDE_THIRD, raw_y1 >= (100 - WIDE_THIRD)
            if (left0 and right1) or (right0 and left1):
                dx = (float(q[Q_END_X]) - e["x"]) / 100 * PITCH_X
                dy = (raw_y1 - raw_y0) / 100 * PITCH_Y
                if math.hypot(dx, dy) >= SWITCH_MIN_DIST_M:
                    is_switch = True

        rows.append({
            "contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"), "playerId": e.get("playerId"),
            "minute": e["timeMin"], "second": e["timeSec"], "period": e["periodId"],
            "eventId": e["eventId"], "x": xm, "y": ym, "end_x": end_x, "end_y": end_y,
            "completed": completed, "is_cross": has_q(e, Q_CROSS), "is_corner": has_q(e, Q_CORNER),
            "is_long_ball": has_q(e, Q_LONG_BALL), "is_cutback": has_q(e, Q_CUTBACK),
            "is_goal_kick": has_q(e, Q_GOAL_KICK), "is_switch": is_switch and completed,
            "progressive": progressive,
            "final_third_entry": completed and start_dist > 35.0 and end_dist is not None and end_dist <= 35.0,
            "box_entry": (completed and end_x is not None and end_x >= 88.5
                          and 13.84 <= end_y <= 54.16 and not (xm >= 88.5 and 13.84 <= ym <= 54.16)),
            "xt_added": xt_added,
        })
    return rows


def build_defensive_actions(events, directions):
    rows = []
    for e in events:
        if e["typeId"] not in DEFENSIVE_TYPES or e.get("x") is None:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"), "minute": e["timeMin"], "period": e["periodId"],
            "x": xm, "y": ym, "action": DEFENSIVE_TYPES[e["typeId"]], "success": e.get("outcome", 1) == 1,
        })
    return rows


def build_pressing_actions(events, directions):
    rows = []
    for e in events:
        if e.get("x") is None or not e.get("contestantId"):
            continue
        t = e["typeId"]
        if t in PRESSING_TYPES:
            action = PRESSING_TYPES[t]
        elif t == T_FOUL and e.get("outcome") == 0:
            action = "Foul"
        else:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"), "minute": e["timeMin"], "period": e["periodId"],
            "x": xm, "y": ym, "action": action,
        })
    return rows


def compute_ppda(passes, pressing_actions, contestant_id, opponent_id, lo=None, hi=None):
    def in_window(m):
        return (lo is None or m >= lo) and (hi is None or m < hi)

    opp_passes = sum(1 for p in passes if p["contestantId"] == opponent_id
                      and in_window(p["minute"]) and p["x"] <= PPDA_ZONE_M)
    def_actions = sum(1 for d in pressing_actions if d["contestantId"] == contestant_id
                       and in_window(d["minute"]) and d["x"] >= (105.0 - PPDA_ZONE_M))
    return opp_passes / def_actions if def_actions else float("nan")


def build_touches(events, directions):
    rows = []
    for e in events:
        if e.get("x") is None or not e.get("contestantId"):
            continue
        if e["typeId"] in (T_SUB_OFF, T_SUB_ON):
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
            "minute": e["timeMin"], "second": e["timeSec"], "period": e["periodId"],
            "x": xm, "y": ym, "typeId": e["typeId"],
        })
    return rows


def build_cards(events):
    rows = []
    for e in events:
        if e["typeId"] != T_CARD:
            continue
        kind = "Red" if has_q(e, Q_RED_CARD) else ("2nd Yellow" if has_q(e, Q_SECOND_YELLOW) else "Yellow")
        rows.append({"contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
                      "player": e.get("playerName", "Unknown"), "minute": e["timeMin"], "kind": kind})
    return rows

### `TeamMatch` + `LeagueMW1`

`TeamMatch(team_id)` loads one team's own matchday-1 fixture. `LeagueMW1` loads
all 14 teams with a matchday-1 feed in this repo — the comparison set for every
"league ranking" page, a single round, not a season.

In [3]:
class TeamMatch:
    """Everything derived from ONE team's matchday-1 event feed: shots,
    passes, defensive/pressing actions, touches, cards, indexable by that
    team's own contestantId or its MW1 opponent's (numbers conceded)."""

    def __init__(self, team_id):
        info = TEAM_MATCH_FILE[team_id]
        self.team_id = team_id
        self.team_name = team_name(team_id)
        self.opponent_id = info["away_id"] if team_id == info["home_id"] else info["home_id"]
        self.opponent_name = team_name(self.opponent_id)
        self.is_home = team_id == info["home_id"]
        self.match_details, self.events = load_match(info["path"])
        self.directions = compute_attack_directions(self.events)
        self.shots = build_shots(self.events, self.directions)
        self.passes = build_passes(self.events, self.directions)
        self.defs = build_defensive_actions(self.events, self.directions)
        self.pressing = build_pressing_actions(self.events, self.directions)
        self.cards = build_cards(self.events)
        self.touches = build_touches(self.events, self.directions)

    def own(self, rows):
        return [r for r in rows if r["contestantId"] == self.team_id]

    def against(self, rows):
        return [r for r in rows if r["contestantId"] == self.opponent_id]

    @property
    def xg_for(self):
        return sum(s["xg"] for s in self.own(self.shots))

    @property
    def xg_against(self):
        return sum(s["xg"] for s in self.against(self.shots))

    def ppda_for(self):
        return compute_ppda(self.passes, self.pressing, self.team_id, self.opponent_id)

    def verticality(self):
        """Avg forward (toward opponent goal) distance per completed forward pass, metres."""
        fwd = [p["end_x"] - p["x"] for p in self.own(self.passes)
               if p["completed"] and p["end_x"] is not None and p["end_x"] > p["x"]]
        return sum(fwd) / len(fwd) if fwd else 0.0

    def def_line_height(self):
        """Avg x of this team's defensive actions while defending -- estimated
        line height, own goal at x=0."""
        d = self.own(self.defs) + [p for p in self.own(self.pressing) if p["action"] != "Foul"]
        return sum(p["x"] for p in d) / len(d) if d else float("nan")

    def halfspace_zone14_counts(self):
        passes = [p for p in self.own(self.passes) if p["completed"] and p["end_x"] is not None]
        zone14 = sum(1 for p in passes if ZONE14[0] <= p["end_x"] < ZONE14[1] and ZONE14[2] <= p["end_y"] < ZONE14[3])
        halfspace = sum(1 for p in passes if any(x0 <= p["end_x"] < x1 and y0 <= p["end_y"] < y1
                                                  for x0, x1, y0, y1 in HALF_SPACES))
        return halfspace, zone14

    def touch_share(self):
        h = len(self.own(self.touches))
        a = len(self.against(self.touches))
        return h / (h + a) if (h + a) else float("nan")

    def field_tilt(self):
        h = sum(1 for t in self.own(self.touches) if t["x"] >= 70)
        a = sum(1 for t in self.against(self.touches) if t["x"] >= 70)
        return h / (h + a) if (h + a) else float("nan")


class LeagueMW1:
    """All 14 teams with a matchday-1 feed in this repo, one TeamMatch each.
    The "league ranking" pages' comparison set -- a single round, not a
    season, and every chart built from it says so."""

    def __init__(self):
        self.teams = {tid: TeamMatch(tid) for tid in TEAM_NAMES}

    def ranking(self, metric_fn, reverse=True):
        vals = [(tid, metric_fn(tm)) for tid, tm in self.teams.items()]
        vals = [(tid, v) for tid, v in vals if v == v]  # drop NaN
        return sorted(vals, key=lambda kv: kv[1], reverse=reverse)

### Quick sanity check

Same checks run at the bottom of `match_data.py` when executed directly.

In [4]:
sparta = TeamMatch(SPARTA_ID)
print(sparta.team_name, "MW1 vs", sparta.opponent_name, "(home)" if sparta.is_home else "(away)")
print("  xG for/against:", round(sparta.xg_for, 2), round(sparta.xg_against, 2))
print("  Verticality:", round(sparta.verticality(), 1), "m/pass")
print("  Def line height:", round(sparta.def_line_height(), 1), "m")
print("  Touch share:", round(sparta.touch_share(), 2), "Field tilt:", round(sparta.field_tilt(), 2))
hs, z14 = sparta.halfspace_zone14_counts()
print("  Halfspace passes:", hs, "Zone14 passes:", z14)
switches = sum(1 for p in sparta.own(sparta.passes) if p["is_switch"])
print("  Switches of play:", switches)
print("  Goal kicks:", sum(1 for p in sparta.own(sparta.passes) if p["is_goal_kick"]))
print("  Cutbacks:", sum(1 for p in sparta.own(sparta.passes) if p["is_cutback"]))

Sparta Praha MW1 vs Zbrojovka Brno (away)
  xG for/against: 0.35 2.77
  Verticality: 10.9 m/pass
  Def line height: 50.4 m
  Touch share: 0.57 Field tilt: 0.64
  Halfspace passes: 73 Zone14 passes: 13
  Switches of play: 5
  Goal kicks: 6
  Cutbacks: 0


## Chart building (`build_charts.py`)

All 20 chart functions, Meridian house style (dark). Each mirrors the visual
grammar of the sibling reports in this repo (pitches via `mplsoccer`,
`housestyle.style` + `housestyle.components` for the header/footer/brand-mark
chrome), but pulls its data from Sparta's own matchday-1 `TeamMatch` and the
14-team `LeagueMW1` sample instead of a full season.

In [5]:
import math
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from mplsoccer import Pitch

sys.path.insert(0, REPO_ROOT)  # NOTEBOOK_DIR/REPO_ROOT resolved in the setup cell above
from housestyle import style, components  # noqa: E402
from housestyle.colors import CATEGORICAL_DARK  # noqa: E402

import match_data as md  # noqa: E402

OUT_DIR = os.path.join(NOTEBOOK_DIR, "Visuals")
os.makedirs(OUT_DIR, exist_ok=True)

FIGSIZE = (13.33, 7.5)
OPP_C = CATEGORICAL_DARK[0]   # ink blue -- Zbrojovka Brno (this match's opponent)
LEAGUE_MUTED = "#5A6672"      # league-context gray, distinct from house axis gray


def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=170, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("Saved:", path)


def new_fig():
    palette, cats = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    return fig, palette


def new_pitch(palette):
    return Pitch(pitch_type="uefa", pitch_color=palette["surface"], line_color=palette["axis"],
                 linewidth=1.0, half=False, line_zorder=2, pad_left=2, pad_right=2)

In [6]:
# ---------------------------------------------------------------------------
# 01. Cover
# ---------------------------------------------------------------------------

def cover(sparta):
    palette, _ = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor(palette["surface"])

    fig.text(0.5, 0.66, "AC SPARTA PRAHA", fontsize=36, fontweight="bold",
              color=palette["accent"], family="serif", ha="center", va="center")
    fig.text(0.5, 0.57, "MATCHDAY 1 ANALYSIS", fontsize=14, fontweight="bold",
              color=palette["ink_secondary"], family="sans-serif", ha="center", va="center", )

    scores = sparta.match_details["scores"]["ft"]
    h_score, a_score = (scores["home"], scores["away"]) if sparta.is_home else (scores["away"], scores["home"])
    fig.text(0.5, 0.44, f"{md.team_name(md.BRNO_ID)}  {scores['home']} – {scores['away']}  Sparta Praha",
              fontsize=17, color=palette["ink_primary"], family="sans-serif", ha="center", va="center")
    fig.text(0.5, 0.385, f"{md.COMPETITION}  ·  {md.VENUE}  ·  {md.MATCH_DATE}", fontsize=11.5,
              color=palette["ink_secondary"], family="sans-serif", ha="center", va="center")

    fig.text(0.5, 0.24, f"{components.MARK} MATCH ANALYSIS  ·  20 PAGES", fontsize=13, fontweight="bold",
              color=palette["accent"], family="sans-serif", ha="center", va="center")
    fig.text(0.5, 0.195, "Rebuilt in Meridian house style from a season-long template -- scoped here to\n"
                         "matchday 1, the only round with an event feed in this repo (14 of 16 teams)",
              fontsize=9.5, color=palette["ink_muted"], family="sans-serif", ha="center", va="center")

    components.brand_mark(fig, palette=palette, right=0.94, y=0.93)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "01_cover.png")


# ---------------------------------------------------------------------------
# 02. Shape: average on-pitch positions
# ---------------------------------------------------------------------------

def shape_average_positions(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.55, 0.62])
    pitch.draw(ax=ax)

    by_player = {}
    for p in sparta.own(sparta.passes):
        by_player.setdefault(p["player"], []).append((p["x"], p["y"]))
    avg = {pl: (np.mean([v[0] for v in vs]), np.mean([v[1] for v in vs]), len(vs))
           for pl, vs in by_player.items() if len(vs) >= 5}
    max_n = max(v[2] for v in avg.values()) if avg else 1
    for pl, (x, y, n) in avg.items():
        size = 300 + 700 * (n / max_n)
        pitch.scatter(x, y, ax=ax, s=size, color=palette["surface"], edgecolors=palette["accent"],
                      linewidth=2.0, zorder=4)
        pitch.annotate(pl.split(" ")[-1], (x, y), ax=ax, ha="center", va="center", fontsize=8.2,
                       color=palette["ink_primary"], fontweight="bold", zorder=5)

    ax2 = fig.add_axes([0.66, 0.18, 0.30, 0.50])
    ax2.axis("off")
    lines = [
        ("Result", "Lost 1-3 away"),
        ("xG created / conceded", f"{sparta.xg_for:.2f} / {sparta.xg_against:.2f}"),
        ("Touch share", f"{sparta.touch_share():.0%}"),
        ("Field tilt (final 3rd)", f"{sparta.field_tilt():.0%}"),
        ("PPDA", f"{sparta.ppda_for():.1f}"),
        ("Verticality", f"{sparta.verticality():.1f} m/pass"),
    ]
    for i, (label, val) in enumerate(lines):
        y = 1.0 - i * 0.16
        ax2.text(0.0, y, label.upper(), fontsize=9, fontweight="bold", color=palette["accent"], va="top")
        ax2.text(0.0, y - 0.06, val, fontsize=13.5, color=palette["ink_primary"], va="top", fontweight="bold")
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1.05)

    components.header(fig, kicker="Shape",
                       title="Sparta Praha: average on-pitch position, matchday 1",
                       dek="Node = avg. pass location (≥5 passes), size = passes played  ·  shown from real "
                           "positions, not a guessed formation label (no verified code lookup for this feed)",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "02_shape_average_positions.png")


# ---------------------------------------------------------------------------
# 03. Passing network
# ---------------------------------------------------------------------------

def _average_positions(team_passes, min_passes=6):
    completed = [p for p in team_passes if p["completed"] and p["end_x"] is not None]
    by_player = {}
    for p in completed:
        by_player.setdefault(p["player"], []).append((p["x"], p["y"]))
    return {pl: (np.mean([v[0] for v in vs]), np.mean([v[1] for v in vs]), len(vs))
            for pl, vs in by_player.items() if len(vs) >= min_passes}


def _combinations(team_passes, avg_pos):
    combos = {}
    ordered = sorted(team_passes, key=lambda p: (p["period"], p["minute"] * 60 + p["second"]))
    for i in range(len(ordered) - 1):
        p, nxt = ordered[i], ordered[i + 1]
        if not p["completed"]:
            continue
        if p["player"] not in avg_pos or nxt["player"] not in avg_pos or p["player"] == nxt["player"]:
            continue
        key = tuple(sorted((p["player"], nxt["player"])))
        combos[key] = combos.get(key, 0) + 1
    return combos


def passing_network(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    color = palette["accent"]
    team_passes = sparta.own(sparta.passes)

    ax = fig.add_axes([0.02, 0.10, 0.54, 0.62])
    avg_pos = _average_positions(team_passes)
    combos = _combinations(team_passes, avg_pos)
    pitch.draw(ax=ax)
    max_c = max(combos.values()) if combos else 1
    for (p1, p2), c in combos.items():
        if c < 2:
            continue
        x1, y1, _ = avg_pos[p1]
        x2, y2, _ = avg_pos[p2]
        pitch.lines(x1, y1, x2, y2, ax=ax, color=color, alpha=0.25 + 0.5 * (c / max_c),
                    lw=0.6 + 3.0 * (c / max_c), zorder=2)
    max_n = max(v[2] for v in avg_pos.values()) if avg_pos else 1
    for pl, (x, y, n) in avg_pos.items():
        size = 260 + 900 * (n / max_n)
        pitch.scatter(x, y, ax=ax, s=size, color=palette["surface"], edgecolors=color,
                      linewidth=2.0, zorder=4)
        pitch.annotate(pl.split(" ")[-1], (x, y), ax=ax, ha="center", va="center", fontsize=8.6,
                       color=palette["ink_primary"], fontweight="bold", zorder=5)

    by_player = {}
    for p in team_passes:
        by_player.setdefault(p["player"], {"att": 0, "comp": 0, "prog": 0, "box": 0})
        d = by_player[p["player"]]
        d["att"] += 1
        d["comp"] += int(p["completed"])
        d["prog"] += int(p["progressive"])
        d["box"] += int(p["box_entry"])

    ax2 = fig.add_axes([0.60, 0.14, 0.37, 0.58])
    ax2.axis("off")
    rows = sorted(by_player.items(), key=lambda kv: -kv[1]["att"])[:14]
    headers = ["Player", "Pass", "Acc%", "Prog", "Box"]
    col_x = [0.0, 0.50, 0.64, 0.80, 0.94]
    for x, h in zip(col_x, headers):
        ax2.text(x, 1.0, h, fontsize=9.5, fontweight="bold", color=palette["ink_primary"], va="top",
                 ha="left" if x == 0 else "center")
    ax2.axhline(0.975, color=palette["axis"], linewidth=0.9)
    row_h = 0.94 / max(len(rows), 1)
    for i, (pl, d) in enumerate(rows):
        y = 0.94 - i * row_h
        acc = d["comp"] / d["att"] if d["att"] else 0
        vals = [pl, str(d["att"]), f"{acc:.0%}", str(d["prog"]), str(d["box"])]
        for x, v in zip(col_x, vals):
            ax2.text(x, y, v, fontsize=8.8, color=palette["ink_secondary"], va="top",
                     ha="left" if x == 0 else "center")
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1.03)

    components.header(fig, kicker="Passing Network",
                       title="Sparta Praha: matchday-1 build-up shape",
                       dek="Average completed-pass position (≥6 passes), full match, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "03_passing_network.png")

In [7]:
# ---------------------------------------------------------------------------
# 04. Ball progression by pitch third
# ---------------------------------------------------------------------------

def ball_progression(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    prog = [p for p in sparta.own(sparta.passes) if p["progressive"]]
    thirds = [("Defensive third", 0, 35), ("Middle third", 35, 70), ("Offensive third", 70, 105.1)]
    axes = [fig.add_axes([0.03 + i * 0.325, 0.36, 0.30, 0.44]) for i in range(3)]
    for ax, (label, lo, hi) in zip(axes, thirds):
        pitch.draw(ax=ax)
        pts = [p for p in prog if lo <= p["x"] < hi]
        xs = [p["x"] for p in pts]
        ys = [p["y"] for p in pts]
        if xs:
            stats = pitch.bin_statistic(xs, ys, statistic="count", bins=(6, 4))
            pitch.heatmap(stats, ax=ax, cmap="Oranges", edgecolors=palette["surface"], alpha=0.9, zorder=1)
        ax.set_title(f"{label} ({len(pts)})", color=palette["ink_primary"], fontsize=11, fontweight="bold",
                     family="sans-serif")

    by_player = {}
    for p in prog:
        by_player[p["player"]] = by_player.get(p["player"], 0) + 1
    top = sorted(by_player.items(), key=lambda kv: -kv[1])[:10]
    ax_tab = fig.add_axes([0.10, 0.10, 0.80, 0.20])
    ax_tab.axis("off")
    n = len(top)
    for i, (pl, c) in enumerate(top):
        x = (i % 5) * 0.2
        y = 1.0 - (i // 5) * 0.5
        ax_tab.text(x, y, f"{pl}", fontsize=9.5, color=palette["ink_primary"], va="top", ha="left", fontweight="bold")
        ax_tab.text(x, y - 0.22, f"{c} progressive passes", fontsize=8.5, color=palette["ink_muted"], va="top", ha="left")
    ax_tab.set_xlim(0, 1)
    ax_tab.set_ylim(0, 1.1)

    components.header(fig, kicker="Progression",
                       title=f"Sparta Praha: {len(prog)} progressive passes, where they started",
                       dek="Origin location of completed progressive passes, by pitch third (own-goal-left convention)",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "04_ball_progression.png")


# ---------------------------------------------------------------------------
# 05. Possession vs field tilt (league scatter)
# ---------------------------------------------------------------------------

def possession_vs_field_tilt(league):
    fig, palette = new_fig()
    ax = fig.add_axes([0.10, 0.16, 0.82, 0.58])

    for tid, tm in league.teams.items():
        x, y = tm.touch_share() * 100, tm.field_tilt() * 100
        is_sparta = tid == md.SPARTA_ID
        ax.scatter([x], [y], s=170 if is_sparta else 60,
                   color=palette["accent"] if is_sparta else LEAGUE_MUTED,
                   edgecolors=palette["ink_primary"] if is_sparta else "none", linewidth=1.6, zorder=5 if is_sparta else 3)
        ax.annotate(tm.team_name, xy=(x, y), xytext=(7, 5), textcoords="offset points",
                    fontsize=9.5 if is_sparta else 8, color=palette["ink_primary"] if is_sparta else palette["ink_muted"],
                    fontweight="bold" if is_sparta else "normal")

    ax.axhline(50, color=palette["axis"], linewidth=0.8, linestyle=":")
    ax.axvline(50, color=palette["axis"], linewidth=0.8, linestyle=":")
    ax.set_xlabel("Touch share this match (%)")
    ax.set_ylabel("Final-third touch share (%)")

    components.header(fig, kicker="Possession",
                       title="Sparta had the ball and the territory, matchday 1",
                       dek="Touch share vs share of final-third touches, all 14 teams with a matchday-1 feed",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "05_possession_vs_field_tilt.png")


# ---------------------------------------------------------------------------
# 06. Progressive pass density (origin vs reception)
# ---------------------------------------------------------------------------

def progressive_pass_density(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax1 = fig.add_axes([0.03, 0.12, 0.44, 0.62])
    ax2 = fig.add_axes([0.53, 0.12, 0.44, 0.62])
    pitch.draw(ax=ax1)
    pitch.draw(ax=ax2)

    prog = [p for p in sparta.own(sparta.passes) if p["progressive"]]
    ox = [p["x"] for p in prog]; oy = [p["y"] for p in prog]
    rx = [p["end_x"] for p in prog]; ry = [p["end_y"] for p in prog]
    stats1 = pitch.bin_statistic(ox, oy, statistic="count", bins=(7, 5))
    stats2 = pitch.bin_statistic(rx, ry, statistic="count", bins=(7, 5))
    pitch.heatmap(stats1, ax=ax1, cmap="Oranges", edgecolors=palette["surface"], alpha=0.92, zorder=1)
    pitch.heatmap(stats2, ax=ax2, cmap="Oranges", edgecolors=palette["surface"], alpha=0.92, zorder=1)
    ax1.set_title("Origin locations", color=palette["ink_primary"], fontsize=12, fontweight="bold", family="sans-serif")
    ax2.set_title("Reception locations", color=palette["ink_primary"], fontsize=12, fontweight="bold", family="sans-serif")

    components.header(fig, kicker="Progressive Passing",
                       title="Sparta Praha: where progressive passes start and land",
                       dek=f"{len(prog)} progressive passes, matchday 1, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "06_progressive_pass_density.png")

In [8]:
# ---------------------------------------------------------------------------
# 07. Switches of play (league ranking + spatial map)
# ---------------------------------------------------------------------------

def switches_of_play(sparta, league):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax1 = fig.add_axes([0.05, 0.14, 0.42, 0.58])
    pitch.draw(ax=ax1)
    sw = [p for p in sparta.own(sparta.passes) if p["is_switch"]]
    for p in sw:
        pitch.arrows(p["x"], p["y"], p["end_x"], p["end_y"], ax=ax1, color=palette["accent"],
                    alpha=0.8, width=2.0, headwidth=6, headlength=6, zorder=4)
    ax1.set_title(f"{len(sw)} switches of play (this match)", color=palette["ink_primary"], fontsize=11.5,
                  fontweight="bold", family="sans-serif")

    ax2 = fig.add_axes([0.55, 0.42, 0.40, 0.30])
    ranking = league.ranking(lambda tm: sum(1 for p in tm.own(tm.passes) if p["is_switch"]))
    vals = [v for _, v in ranking]
    ypos_sparta = next(i for i, (tid, _) in enumerate(ranking) if tid == md.SPARTA_ID)
    xs = np.arange(len(ranking))
    colors = [palette["accent"] if tid == md.SPARTA_ID else LEAGUE_MUTED for tid, _ in ranking]
    ax2.bar(xs, vals, color=colors)
    ax2.set_xticks([])
    ax2.set_ylabel("Switches")
    ax2.set_title(f"League rank: #{ypos_sparta + 1} of {len(ranking)} (matchday 1)", color=palette["ink_primary"],
                  fontsize=10.5, fontweight="bold", family="sans-serif")

    by_player = {}
    for p in sw:
        by_player[p["player"]] = by_player.get(p["player"], 0) + 1
    ax3 = fig.add_axes([0.55, 0.14, 0.40, 0.22])
    top = sorted(by_player.items(), key=lambda kv: -kv[1])
    ypos = np.arange(len(top))[::-1]
    ax3.barh(ypos, [v for _, v in top], color=palette["accent"])
    ax3.set_yticks(ypos)
    ax3.set_yticklabels([k for k, _ in top], fontsize=9.5)
    ax3.set_xlabel("Switches completed")

    components.header(fig, kicker="Switches Of Play",
                       title="Sparta Praha's flank-to-flank passing, matchday 1",
                       dek="Completed open-play pass, ≥30m, wide channel to wide channel  ·  ranked among the "
                           "14 teams with a matchday-1 feed, not a season table",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "07_switches_of_play.png")


# ---------------------------------------------------------------------------
# 08. Creative zone analysis (halfspace / zone 14)
# ---------------------------------------------------------------------------

def creative_zone_analysis(sparta, league):
    fig, palette = new_fig()
    ax1 = fig.add_axes([0.06, 0.16, 0.40, 0.30])
    ax2 = fig.add_axes([0.56, 0.16, 0.40, 0.30])

    hs_rank = league.ranking(lambda tm: tm.halfspace_zone14_counts()[0])
    z14_rank = league.ranking(lambda tm: tm.halfspace_zone14_counts()[1])
    for ax, rank, label in ((ax1, hs_rank, "Half-space passes"), (ax2, z14_rank, "Zone-14 passes")):
        xs = np.arange(len(rank))
        colors = [palette["accent"] if tid == md.SPARTA_ID else LEAGUE_MUTED for tid, _ in rank]
        ax.bar(xs, [v for _, v in rank], color=colors)
        ax.set_xticks([])
        ax.set_title(label, color=palette["ink_primary"], fontsize=11.5, fontweight="bold", family="sans-serif")
        sparta_rank = next(i for i, (tid, _) in enumerate(rank) if tid == md.SPARTA_ID)
        ax.set_xlabel(f"Sparta: #{sparta_rank + 1} of {len(rank)}")

    def top_players(zone_check):
        counts = {}
        for p in sparta.own(sparta.passes):
            if p["completed"] and p["end_x"] is not None and zone_check(p["end_x"], p["end_y"]):
                counts[p["player"]] = counts.get(p["player"], 0) + 1
        return sorted(counts.items(), key=lambda kv: -kv[1])[:8]

    hs_top = top_players(lambda x, y: any(x0 <= x < x1 and y0 <= y < y1 for x0, x1, y0, y1 in md.HALF_SPACES))
    z14_top = top_players(lambda x, y: md.ZONE14[0] <= x < md.ZONE14[1] and md.ZONE14[2] <= y < md.ZONE14[3])

    ax3 = fig.add_axes([0.14, 0.52, 0.32, 0.24])
    ax4 = fig.add_axes([0.64, 0.52, 0.32, 0.24])
    for ax, top in ((ax3, hs_top), (ax4, z14_top)):
        ypos = np.arange(len(top))[::-1]
        ax.barh(ypos, [v for _, v in top], color=palette["accent"])
        ax.set_yticks(ypos)
        ax.set_yticklabels([k for k, _ in top], fontsize=9)

    components.header(fig, kicker="Creative Zones",
                       title="Sparta Praha's half-space and zone-14 receptions, matchday 1",
                       dek="Completed-pass reception counts  ·  league bars = 14-team matchday-1 sample",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "08_creative_zone_analysis.png")


# ---------------------------------------------------------------------------
# 09. Box entries (swapped in for the source deck's cutback page -- Sparta's
# own matchday-1 fixture has zero cutback events; see match_data.py)
# ---------------------------------------------------------------------------

def box_entries_map(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.90, 0.62])
    pitch.draw(ax=ax)

    entries = [p for p in sparta.own(sparta.passes) if p["box_entry"]]
    for p in entries:
        is_cross = p["is_cross"]
        pitch.arrows(p["x"], p["y"], p["end_x"], p["end_y"], ax=ax,
                    color=palette["accent"] if is_cross else CATEGORICAL_DARK[2],
                    alpha=0.85, width=2.0, headwidth=6, headlength=6, zorder=4)

    legend_elems = [Line2D([0], [0], color=palette["accent"], lw=2.2, label="Cross into box"),
                    Line2D([0], [0], color=CATEGORICAL_DARK[2], lw=2.2, label="Other pass into box")]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.05), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Box Entries",
                       title=f"Sparta Praha: {len(entries)} completed passes into the box, matchday 1",
                       dek="Swapped in for the source template's cutback map -- Sparta's own matchday-1 fixture "
                           "recorded zero cutbacks (rare leaguewide at this sample size, see match_data.py)",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "09_box_entries_map.png")


# ---------------------------------------------------------------------------
# 10. Long ball targets
# ---------------------------------------------------------------------------

def long_ball_targets(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.03, 0.10, 0.55, 0.62])
    pitch.draw(ax=ax)

    lb = [p for p in sparta.own(sparta.passes) if p["is_long_ball"] and p["completed"] and p["end_x"] is not None]
    by_player = {}
    for p in lb:
        by_player.setdefault(p["player"], []).append(p)
    for pl, pts in by_player.items():
        xs = [p["end_x"] for p in pts]; ys = [p["end_y"] for p in pts]
        pitch.scatter(xs, ys, ax=ax, s=90 + 30 * len(pts), color=palette["accent"], alpha=0.7,
                      edgecolors=palette["surface"], linewidth=0.6, zorder=4)

    top = sorted(by_player.items(), key=lambda kv: -len(kv[1]))[:8]
    ax2 = fig.add_axes([0.63, 0.16, 0.33, 0.50])
    ypos = np.arange(len(top))[::-1]
    ax2.barh(ypos, [len(v) for _, v in top], color=palette["accent"])
    ax2.set_yticks(ypos)
    ax2.set_yticklabels([k for k, _ in top], fontsize=10)
    ax2.set_xlabel("Completed long balls received")

    components.header(fig, kicker="Long Balls",
                       title=f"Sparta Praha: {len(lb)} completed long balls, matchday 1",
                       dek="Reception locations, own goal on the left, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "10_long_ball_targets.png")

In [9]:
# ---------------------------------------------------------------------------
# 11. Goal kick end locations
# ---------------------------------------------------------------------------

def goal_kick_end_locations(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.90, 0.62])
    pitch.draw(ax=ax)

    gks = [p for p in sparta.own(sparta.passes) if p["is_goal_kick"] and p["end_x"] is not None]
    short = [p for p in gks if math.hypot(p["end_x"] - p["x"], p["end_y"] - p["y"]) < 30]
    long_ = [p for p in gks if math.hypot(p["end_x"] - p["x"], p["end_y"] - p["y"]) >= 30]
    if short:
        pitch.scatter([p["end_x"] for p in short], [p["end_y"] for p in short], ax=ax, s=140,
                      color=CATEGORICAL_DARK[2], edgecolors=palette["surface"], linewidth=0.8,
                      alpha=0.85, zorder=4, label=f"Short (<30m), {len(short)}")
    if long_:
        pitch.scatter([p["end_x"] for p in long_], [p["end_y"] for p in long_], ax=ax, s=140,
                      color=palette["accent"], edgecolors=palette["surface"], linewidth=0.8,
                      alpha=0.85, zorder=4, label=f"Long (≥30m), {len(long_)}")
    fig.legend(loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.05),
               fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Goal Kicks",
                       title=f"Sparta Praha: {len(gks)} goal kicks, matchday 1",
                       dek="End location of each goal kick, own goal on the left, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "11_goal_kick_end_locations.png")


# ---------------------------------------------------------------------------
# 12-13. League ranking bars: halfspace receptions (middle third / final third)
# ---------------------------------------------------------------------------

def _league_rank_bar(league, metric_fn, page_num, kicker, title, dek, fname):
    fig, palette = new_fig()
    ax = fig.add_axes([0.22, 0.12, 0.70, 0.62])
    ranking = league.ranking(metric_fn)
    ypos = np.arange(len(ranking))[::-1]
    colors = [palette["accent"] if tid == md.SPARTA_ID else palette["axis"] for tid, _ in ranking]
    ax.barh(ypos, [v for _, v in ranking], color=colors)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"#{i+1}  {md.team_name(tid)}" for i, (tid, _) in enumerate(ranking)], fontsize=10)
    for y, (tid, v) in zip(ypos, ranking):
        weight = "bold" if tid == md.SPARTA_ID else "normal"
        ax.text(v + max(v2 for _, v2 in ranking) * 0.015, y, f"{v:.1f}" if isinstance(v, float) else str(v),
                va="center", fontsize=9.5, color=palette["ink_primary"], fontweight=weight)
    avg = sum(v for _, v in ranking) / len(ranking)
    ax.axvline(avg, color=palette["ink_muted"], linewidth=1.0, linestyle="--")
    ax.text(avg, len(ranking) - 0.3, " sample avg", fontsize=8, color=palette["ink_muted"], va="bottom")

    components.header(fig, kicker=kicker, title=title, dek=dek, palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_{fname}.png")


def middle_third_halfspace_ranking(league):
    def metric(tm):
        passes = [p for p in tm.own(tm.passes) if p["completed"] and p["end_x"] is not None and 35 <= p["end_x"] < 70]
        return sum(1 for p in passes if any(x0 <= p["end_x"] < x1 and y0 <= p["end_y"] < y1 for x0, x1, y0, y1 in md.HALF_SPACES))
    _league_rank_bar(league, metric, "12", "League Ranking",
                      "Middle-third half-space receptions, matchday 1",
                      "Completed-pass receptions in the middle-third half-spaces  ·  14-team matchday-1 sample",
                      "middle_third_halfspace")


def final_third_halfspace_ranking(league):
    def metric(tm):
        passes = [p for p in tm.own(tm.passes) if p["completed"] and p["end_x"] is not None and p["end_x"] >= 70]
        return sum(1 for p in passes if any(x0 <= p["end_x"] < x1 and y0 <= p["end_y"] < y1 for x0, x1, y0, y1 in md.HALF_SPACES))
    _league_rank_bar(league, metric, "13", "League Ranking",
                      "Final-third half-space receptions, matchday 1",
                      "Completed-pass receptions in the final-third half-spaces  ·  14-team matchday-1 sample",
                      "final_third_halfspace")


def verticality_ranking(league):
    _league_rank_bar(league, lambda tm: tm.verticality(), "14", "League Ranking",
                      "Team verticality, matchday 1",
                      "Avg forward distance (m) per completed forward pass  ·  14-team matchday-1 sample",
                      "verticality")


def def_line_height_ranking(league):
    _league_rank_bar(league, lambda tm: tm.def_line_height(), "17", "League Ranking",
                      "Average defensive line height, matchday 1",
                      "Estimated from x-location of defensive/pressing actions  ·  14-team matchday-1 sample",
                      "def_line_height")


# ---------------------------------------------------------------------------
# 15. Attacking sequence involvements
# ---------------------------------------------------------------------------

def attacking_sequence_involvements(sparta):
    fig, palette = new_fig()
    ax = fig.add_axes([0.20, 0.16, 0.72, 0.58])

    shots_by = {}
    for s in sparta.own(sparta.shots):
        shots_by[s["player"]] = shots_by.get(s["player"], 0) + 1
    prog_by = {}
    for p in sparta.own(sparta.passes):
        if p["progressive"] or p["box_entry"]:
            prog_by[p["player"]] = prog_by.get(p["player"], 0) + 1

    players = sorted(set(shots_by) | set(prog_by), key=lambda pl: -(shots_by.get(pl, 0) * 3 + prog_by.get(pl, 0)))[:12]
    ypos = np.arange(len(players))[::-1]
    shot_vals = [shots_by.get(pl, 0) for pl in players]
    prog_vals = [prog_by.get(pl, 0) for pl in players]
    ax.barh(ypos, shot_vals, color=palette["accent"], label="Shots")
    ax.barh(ypos, prog_vals, left=shot_vals, color=CATEGORICAL_DARK[2], label="Progressive pass / box entry")
    ax.set_yticks(ypos)
    ax.set_yticklabels(players, fontsize=10.5)
    ax.set_xlabel("Involvements, matchday 1")
    ax.legend(loc="lower right", frameon=False, fontsize=9.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Attacking Involvement",
                       title="Sparta Praha: who drove the attack, matchday 1",
                       dek="Shots plus progressive passes / box entries per player, this match only",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "15_attacking_sequence_involvements.png")

In [10]:
# ---------------------------------------------------------------------------
# 16. Shot conversion vs xG per shot (league scatter)
# ---------------------------------------------------------------------------

def shot_conversion_vs_xg(league):
    fig, palette = new_fig()
    ax = fig.add_axes([0.10, 0.16, 0.82, 0.58])

    for tid, tm in league.teams.items():
        shots = tm.own(tm.shots)
        if not shots:
            continue
        conv = sum(1 for s in shots if s["is_goal"]) / len(shots) * 100
        xg_shot = sum(s["xg"] for s in shots) / len(shots)
        is_sparta = tid == md.SPARTA_ID
        ax.scatter([xg_shot], [conv], s=170 if is_sparta else 60,
                   color=palette["accent"] if is_sparta else LEAGUE_MUTED,
                   edgecolors=palette["ink_primary"] if is_sparta else "none", linewidth=1.6,
                   zorder=5 if is_sparta else 3)
        ax.annotate(tm.team_name, xy=(xg_shot, conv), xytext=(7, 5), textcoords="offset points",
                    fontsize=9.5 if is_sparta else 8, color=palette["ink_primary"] if is_sparta else palette["ink_muted"],
                    fontweight="bold" if is_sparta else "normal")

    ax.set_xlabel("xG per shot (own model)")
    ax.set_ylabel("Shot conversion (%)")

    components.header(fig, kicker="Finishing",
                       title="Shot conversion vs chance quality, matchday 1",
                       dek="Non-penalty shots, all 14 teams with a matchday-1 feed  ·  own xG model, not a season sample",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "16_shot_conversion_vs_xg.png")


# ---------------------------------------------------------------------------
# 18. Shot map
# ---------------------------------------------------------------------------

def shot_map(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.90, 0.62])
    pitch.draw(ax=ax)

    for s in sparta.own(sparta.shots):
        size = 90 + s["xg"] * 900
        if s["is_goal"]:
            pitch.scatter(s["x"], s["y"], ax=ax, s=size, marker="o", color=palette["accent"],
                          edgecolors=palette["ink_primary"], linewidth=1.4, zorder=5)
        else:
            pitch.scatter(s["x"], s["y"], ax=ax, s=size, marker="o", facecolors="none",
                          edgecolors=palette["accent"], linewidth=1.6, alpha=0.85, zorder=4)
    ax.text(0.02, -0.06, "Hollow = shot   ● Filled = goal   Size = xG", transform=ax.transAxes,
            fontsize=8.5, color=palette["ink_muted"])

    xg = sparta.xg_for
    shots = sparta.own(sparta.shots)
    components.header(fig, kicker="Shot Map",
                       title=f"Sparta Praha: {len(shots)} shots, {xg:.2f} xG, matchday 1",
                       dek="Own xG model: distance + angle to goal, header penalty applied",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "18_shot_map.png")


# ---------------------------------------------------------------------------
# 19. Defending: pressing + defensive actions
# ---------------------------------------------------------------------------

def defending_overview(sparta):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.03, 0.10, 0.55, 0.62])
    pitch.draw(ax=ax)
    markers = {"Tackle": "o", "Interception": "D", "Clearance": "s"}
    action_colors = {"Tackle": CATEGORICAL_DARK[2], "Interception": CATEGORICAL_DARK[3],
                      "Clearance": palette["ink_muted"]}
    team_defs = sparta.own(sparta.defs)
    for action, marker in markers.items():
        pts = [d for d in team_defs if d["action"] == action]
        if not pts:
            continue
        pitch.scatter([p["x"] for p in pts], [p["y"] for p in pts], ax=ax, s=80, marker=marker,
                      color=action_colors[action], edgecolors=palette["surface"], linewidth=0.6,
                      alpha=0.9, zorder=4)
    legend_elems = [Line2D([0], [0], marker=markers[a], color=palette["surface"], markerfacecolor=action_colors[a],
                            markersize=10, label=a, linewidth=0) for a in markers]
    fig.legend(handles=legend_elems, loc="lower center", ncol=3, frameon=False,
               bbox_to_anchor=(0.30, 0.05), fontsize=10, labelcolor=palette["ink_secondary"])

    ax2 = fig.add_axes([0.64, 0.16, 0.32, 0.52])
    ax2.axis("off")
    counts = {a: sum(1 for d in team_defs if d["action"] == a) for a in markers}
    lines = [
        ("PPDA", f"{sparta.ppda_for():.1f}"),
        ("Tackles", str(counts["Tackle"])),
        ("Interceptions", str(counts["Interception"])),
        ("Clearances", str(counts["Clearance"])),
        ("Def. line height", f"{sparta.def_line_height():.1f} m"),
        ("xG conceded", f"{sparta.xg_against:.2f}"),
    ]
    for i, (label, val) in enumerate(lines):
        y = 1.0 - i * 0.16
        ax2.text(0.0, y, label.upper(), fontsize=9, fontweight="bold", color=palette["accent"], va="top")
        ax2.text(0.0, y - 0.06, val, fontsize=14, color=palette["ink_primary"], va="top", fontweight="bold")
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1.05)

    components.header(fig, kicker="Defending",
                       title="Sparta Praha: defensive actions, matchday 1",
                       dek="Own goal on the left, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "19_defending_overview.png")

In [11]:
# ---------------------------------------------------------------------------
# 20. Report card (closing summary)
# ---------------------------------------------------------------------------

def report_card(sparta):
    palette, _ = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor(palette["surface"])

    scores = sparta.match_details["scores"]["ft"]
    fig.text(0.5, 0.90, f"{md.team_name(md.BRNO_ID)}  {scores['home']}-{scores['away']}  Sparta Praha",
              fontsize=18, fontweight="bold", color=palette["ink_primary"], family="serif",
              ha="center", va="center")
    fig.text(0.5, 0.855, f"{md.COMPETITION}  ·  {md.VENUE}  ·  {md.MATCH_DATE}",
              fontsize=10.5, color=palette["ink_secondary"], ha="center", va="center")

    passes = sparta.own(sparta.passes)
    rows = [
        ("xG created", f"{sparta.xg_for:.2f}"),
        ("xG conceded", f"{sparta.xg_against:.2f}"),
        ("Touch share", f"{sparta.touch_share():.0%}"),
        ("Pass accuracy", f"{sum(1 for p in passes if p['completed']) / len(passes):.0%}"),
        ("PPDA", f"{sparta.ppda_for():.1f}"),
        ("Verticality", f"{sparta.verticality():.1f} m/pass"),
    ]

    ax = fig.add_axes([0.20, 0.20, 0.60, 0.55])
    ax.axis("off")
    n = len(rows)
    for i, (label, val) in enumerate(rows):
        y = 0.85 - i * (0.85 / n)
        ax.text(0.0, y, label, fontsize=12, color=palette["ink_muted"], ha="left", va="top")
        ax.text(1.0, y, val, fontsize=15, fontweight="bold", color=palette["accent"], ha="right", va="top")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    components.brand_mark(fig, palette=palette, right=0.94, y=0.965)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "20_report_card.png")

## Generate all 20 pages and compile the PDF

In [12]:
def main():
    sparta = md.TeamMatch(md.SPARTA_ID)
    league = md.LeagueMW1()

    cover(sparta)
    shape_average_positions(sparta)
    passing_network(sparta)
    ball_progression(sparta)
    possession_vs_field_tilt(league)
    progressive_pass_density(sparta)
    switches_of_play(sparta, league)
    creative_zone_analysis(sparta, league)
    box_entries_map(sparta)
    long_ball_targets(sparta)
    goal_kick_end_locations(sparta)
    middle_third_halfspace_ranking(league)
    final_third_halfspace_ranking(league)
    verticality_ranking(league)
    attacking_sequence_involvements(sparta)
    shot_conversion_vs_xg(league)
    def_line_height_ranking(league)
    shot_map(sparta)
    defending_overview(sparta)
    report_card(sparta)
    print("Done.")

In [13]:
main()  # build_charts.main() -- generates all 20 PNGs into ./Visuals

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/01_cover.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/02_shape_average_positions.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/03_passing_network.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/04_ball_progression.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/05_possession_vs_field_tilt.png


...

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/16_shot_conversion_vs_xg.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/17_def_line_height.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/18_shot_map.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/19_defending_overview.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Visuals/20_report_card.png
Done.


## Compile the PDF (`build_pdf.py`)

In [14]:
import glob
import os

from PIL import Image
from reportlab.lib.pagesizes import landscape
from reportlab.pdfgen import canvas

OUT_DIR = NOTEBOOK_DIR
VIS_DIR = os.path.join(OUT_DIR, "Visuals")
PDF_PATH = os.path.join(OUT_DIR, "Sparta_Praha_Matchday1_Analysis.pdf")

PAGE_W, PAGE_H = 1920, 1080  # points, 16:9


def build_pdf():
    pages = sorted(glob.glob(os.path.join(VIS_DIR, "*.png")))
    if not pages:
        raise SystemExit("No PNGs found in Visuals/ -- run build_charts.py first")

    c = canvas.Canvas(PDF_PATH, pagesize=landscape((PAGE_H, PAGE_W)))
    for path in pages:
        img = Image.open(path)
        iw, ih = img.size
        scale = min(PAGE_W / iw, PAGE_H / ih)
        w, h = iw * scale, ih * scale
        x, y = (PAGE_W - w) / 2, (PAGE_H - h) / 2
        c.drawImage(path, x, y, width=w, height=h)
        c.showPage()
    c.save()
    print("Saved:", PDF_PATH, f"({len(pages)} pages)")

In [15]:
build_pdf()  # compiles Visuals/*.png into the landscape PDF

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Sparta Praha Matchday 1 Analysis/Sparta_Praha_Matchday1_Analysis.pdf (20 pages)
